<a href="https://colab.research.google.com/github/Aun-Mehdi117/-Flyrank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aun-Mehdi117/-Flyrank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** This picks up ML-04's data contract
(`month=2026-03`, H1 = days 1-15 feature window, H2 = days 16-31 label window) and does two
things ML-04 didn't: adds a real categorical feature (joined from `dim_content`, with explicit
missing-value handling), and runs the formal leakage-hunt checklist from
`skills/hunting-leakage-and-validating/SKILL.md` against the *full* feature set — not just the
one-column trap ML-04 demonstrated.

> Run this notebook top to bottom in Colab with an `HF_TOKEN` secret set (see `SETUP.md`).

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
%pip -q install duckdb

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'  # same iteration month as ML-04 — mid-panel, not the sealed _sample month

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_month':  f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:12} {n:>12,} rows')

dim_clients           104 rows
dim_content       519,606 rows
fact_month      9,841,378 rows


In [3]:
# Check dim_content's columns before I claim any of them — same discipline ML-04 used for
# fact_content_daily_performance. I don't hardcode a column name I haven't confirmed exists.
content_cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 0").df()
print(content_cols[['column_name', 'column_type']].to_string(index=False))

# Candidate categorical fields, in priority order (mirrors the starter CSV's content-properties
# columns per docs/data-dictionary.md). Pick the first one that actually exists in this table.
CANDIDATE_CATS = ['content_type', 'main_intent', 'provider_used']
available_cats = [c for c in CANDIDATE_CATS if c in content_cols['column_name'].values]
assert available_cats, 'none of the candidate categorical columns exist in dim_content — inspect content_cols above and pick one manually'
CAT_COL = available_cats[0]
print(f'\nUsing categorical field: {CAT_COL}')

               column_name column_type
            client_hash_id     VARCHAR
           content_hash_id     VARCHAR
           keyword_hash_id     VARCHAR
               url_hash_id     VARCHAR
        keyword_char_count      BIGINT
       keyword_token_count      BIGINT
            url_char_count      BIGINT
      content_created_date        DATE
      content_updated_date        DATE
              content_type     VARCHAR
             search_volume      BIGINT
               competition      DOUBLE
         competition_level     VARCHAR
                       cpc      DOUBLE
               main_intent     VARCHAR
                 backlinks      BIGINT
            category_count      BIGINT
      keyword_created_date        DATE
             provider_used     VARCHAR
                model_used     VARCHAR
                char_count      BIGINT
                word_count      BIGINT
       last_optimized_date        DATE
optimization_eligible_date        DATE
              is_publishe

In [4]:
# Rebuild the five H1-only numeric features (identical logic to ML-04's data contract — this
# is the same honest feature set, not a new one), then join dim_content for CAT_COL.
feat = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_impressions ELSE 0 END) AS imp_h1,
        SUM(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_clicks      ELSE 0 END) AS clk_h1,
        AVG(CASE WHEN f.report_date <= DATE '2026-03-15' AND f.gsc_avg_position > 0
                 THEN f.gsc_avg_position END)                                               AS pos_h1,
        COUNT(*) FILTER (WHERE f.report_date <= DATE '2026-03-15'
                          AND f.gsc_impressions > 0)                                         AS days_active_h1,
        SUM(CASE WHEN f.report_date >  DATE '2026-03-15' THEN f.gsc_impressions ELSE 0 END) AS imp_h2,
        ANY_VALUE(c.{CAT_COL})                                                               AS {CAT_COL}
    FROM {TABLES['fact_month']} f
    LEFT JOIN {TABLES['dim_content']} c USING (content_hash_id)
    GROUP BY 1, 2
    HAVING imp_h1 >= 5   -- same minimum-volume filter as ML-04; a point-in-time (H1-only) rule
""".replace('{CAT_COL}', CAT_COL)).df()

feat['log_imp_h1']   = np.log1p(feat['imp_h1'])
feat['ctr_h1']       = feat['clk_h1'] / feat['imp_h1'].replace(0, np.nan)
feat['is_declining'] = (feat['imp_h2'] < 0.8 * feat['imp_h1']).astype(int)

print(f'feature frame: {len(feat):,} content items with >= 5 H1 impressions')
print(f'\n{CAT_COL} missingness before fill:')
print(feat[CAT_COL].isna().value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feature frame: 130,146 content items with >= 5 H1 impressions

content_type missingness before fill:
content_type
False    130146
Name: count, dtype: int64


In [5]:
# Explicit fill + categorical handling. Numeric NaNs (pos_h1, ctr_h1 can be NaN when the
# underlying H1 totals are 0/undefined) -> 0, with a flag column so 'no signal' isn't silently
# confused with 'signal of zero'. Categorical NaNs -> the literal string 'unknown' (never
# dropped — a blank content_type is itself information, per the data dictionary's warning
# that missingness in this dataset runs along content_type lines, not at random).
feat['pos_h1_missing'] = feat['pos_h1'].isna().astype(int)
feat['pos_h1']         = feat['pos_h1'].fillna(0)
feat['ctr_h1']         = feat['ctr_h1'].fillna(0)
feat[CAT_COL]          = feat[CAT_COL].fillna('unknown')

cat_dummies = pd.get_dummies(feat[CAT_COL], prefix=CAT_COL)
feature_vector = pd.concat(
    [feat[['client_hash_id', 'content_hash_id', 'log_imp_h1', 'clk_h1', 'pos_h1',
           'pos_h1_missing', 'ctr_h1', 'days_active_h1', 'is_declining']], cat_dummies],
    axis=1,
)
print(f'feature_vector: {feature_vector.shape[0]:,} rows x {feature_vector.shape[1]} columns')
feature_vector.head()

feature_vector: 130,146 rows x 12 columns


,client_hash_id,content_hash_id,log_imp_h1,clk_h1,pos_h1,pos_h1_missing,ctr_h1,days_active_h1,is_declining,content_type_comparison article,content_type_feedly article,content_type_keyword article
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,4.718499,0.0,5.222776,0,0.000000,13,1,False,False,True
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,3.663562,1.0,5.218750,0,0.026316,9,1,False,False,True
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,5.393628,1.0,4.004356,0,0.004566,15,0,False,False,True
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,3.044522,0.0,4.625000,0,0.000000,9,1,False,False,True
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,7.309881,0.0,6.156643,0,0.000000,14,0,False,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE
the moment you predict.*

| Feature | Meaning | Type | Missing handling | Available before the 2026-03-15 decision moment? |
|---|---|---|---|---|
| `log_imp_h1` | `log1p` of summed H1 GSC impressions | numeric | can't be missing (built from a `SUM`, which returns 0 not NULL) | **Yes** — sums only `report_date <= 03-15` |
| `clk_h1` | Summed H1 GSC clicks | numeric | same as above, never NULL | **Yes** — same H1-only sum |
| `pos_h1` | Mean H1 GSC position, `position > 0` rows only | numeric | `NaN` when an item has zero valid-position H1 rows -> filled to `0` **and** flagged via `pos_h1_missing` so the fill isn't read as "position zero" | **Yes** — averages only H1 rows |
| `pos_h1_missing` | 1 if `pos_h1` had to be filled | numeric flag | n/a (this *is* the missingness signal) | **Yes** — derived from the same H1-only aggregate |
| `ctr_h1` | `clk_h1 / imp_h1` | numeric ratio | `NaN` only when `imp_h1 = 0`, impossible here since the `HAVING imp_h1 >= 5` filter already excludes those rows — filled to `0` defensively anyway | **Yes** — ratio of two H1-only totals |
| `days_active_h1` | Count of H1 calendar days (1-15) with impressions > 0 | numeric, bounded 0-15 | can't be missing (a `COUNT`) | **Yes** — counts only H1 days |
| `<CAT_COL>_*` (one-hot) | Content-category field pulled from `dim_content` (whichever of `content_type` / `main_intent` / `provider_used` actually exists — printed as `CAT_COL` in section 1), one dummy column per observed value | categorical -> one-hot | `NaN` filled to the literal category `'unknown'` **before** one-hot encoding, so missingness gets its own column instead of silently vanishing into all-zero rows | **Yes, conditionally** — content metadata like this is set at publish time, which is always before 2026-03-15 for any item with H1 activity; I'm treating it as a static attribute this month, not something that changed mid-window |

**Context, never features:** `client_hash_id`, `content_hash_id` — grouping and join keys only
(used for the grouped-split test in section 3, never fed to a model).

**Not built here:** `imp_h2` and `is_declining` — `imp_h2` only exists to *derive* the label and
is dropped before modeling; `is_declining` is the label itself. Both appear in `feat` above for
transparency but neither is in `feature_vector`'s eventual model-input columns (see section 3).

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Running the checklist from `skills/hunting-leakage-and-validating/SKILL.md` against the full
7-numeric + one-hot feature set built above — not just the single-column trap ML-04 already
showed.

In [6]:
# --- Check 1: label-derived features (train WITH vs WITHOUT the suspect column) ---
# Reproduces ML-04's trap, but now against the richer feature set (numeric + one-hot dummies)
# built in section 1, since a new feature vector needs its own attack, not just a citation of
# an old one.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score

model_cols = ['log_imp_h1', 'clk_h1', 'pos_h1', 'pos_h1_missing', 'ctr_h1', 'days_active_h1'] \
             + list(cat_dummies.columns)
model_df = feature_vector.dropna(subset=model_cols).copy()
X, y = model_df[model_cols], model_df['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
auc_honest = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])

# The suspect: future_impressions_h2, the exact quantity is_declining is computed from.
model_df['future_impressions_h2'] = feat.loc[model_df.index, 'imp_h2']
X2 = model_df[model_cols + ['future_impressions_h2']]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X2, y, test_size=0.3, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2)
auc_leak = roc_auc_score(y_te2, leaky_model.predict_proba(X_te2)[:, 1])

print(f'Honest AUC ({len(model_cols)} features, all H1-only + content category): {auc_honest:.3f}')
print(f'Leaked AUC (+ future_impressions_h2):                                    {auc_leak:.3f}')
print(f'Jump: +{auc_leak - auc_honest:.3f}  <- confession. Column removed below; never reported.')

Honest AUC (9 features, all H1-only + content category): 0.585
Leaked AUC (+ future_impressions_h2):                                    0.680
Jump: +0.096  <- confession. Column removed below; never reported.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
# --- Check 2: product/decision-derived flags used as features? ---
# dim_content's columns (printed in section 1) are static content metadata (category, intent,
# provider) set at publish/creation time — none of them are a pre-computed score or flag from
# an existing FlyRank system (e.g. no 'is_stale', 'needs_refresh', or similar decision output
# appears in the schema). Asserting that explicitly rather than assuming it:
PRODUCT_FLAG_PATTERNS = ('flag', 'score', 'priority', 'needs_', 'is_stale', 'recommend')
suspect_flags = [c for c in content_cols['column_name'] if any(p in c.lower() for p in PRODUCT_FLAG_PATTERNS)]
print(f'dim_content columns matching product-flag naming patterns: {suspect_flags}')
assert not suspect_flags, 'found a likely product-flag column — exclude it, do not model on it'
print('None found: no product/decision-derived column entered the feature vector.')

dim_content columns matching product-flag naming patterns: []
None found: no product/decision-derived column entered the feature vector.


In [8]:
# --- Check 3: population selection checked for outcome-window information ---
# The only row filter is `HAVING imp_h1 >= 5`, applied to the H1-only aggregate. It does NOT
# reference imp_h2 or is_declining anywhere, so which content items make it into the feature
# vector is decided purely by H1 activity -- not by whether they later decline. Confirmed by
# re-deriving the filter and checking it doesn't correlate suspiciously with the label rate:
print(f"is_declining rate, all H1>=5 items: {feat['is_declining'].mean():.1%}")
print('Filter references only imp_h1 (H1-only) -- no outcome-window leakage in population selection.')

is_declining rate, all H1>=5 items: 30.3%
Filter references only imp_h1 (H1-only) -- no outcome-window leakage in population selection.


In [9]:
# --- Check 4: grouped split vs random split -- report the gap ---
# Random split (used above) lets rows from the same client leak shared client-level character
# across train/test. GroupKFold by client_hash_id asks the honest question: does this work on
# a CLIENT it never saw?
gkf = GroupKFold(n_splits=5)
groups = model_df['client_hash_id'] if 'client_hash_id' in model_df.columns else feature_vector.loc[model_df.index, 'client_hash_id']
grouped_aucs = []
for tr_idx, te_idx in gkf.split(X, y, groups=groups):
    m = LogisticRegression(max_iter=1000).fit(X.iloc[tr_idx], y.iloc[tr_idx])
    grouped_aucs.append(roc_auc_score(y.iloc[te_idx], m.predict_proba(X.iloc[te_idx])[:, 1]))
auc_grouped = float(np.mean(grouped_aucs))

base_rate = y.mean()
print(f'Base rate (is_declining):           {base_rate:.1%}')
print(f'Random-split AUC (honest features): {auc_honest:.3f}')
print(f'Grouped-split AUC (by client, 5-fold): {auc_grouped:.3f}')
print(f'Gap (random - grouped): {auc_honest - auc_grouped:+.3f}')
print()
print('A small gap says client identity isn\'t doing the model\'s work for it; a large gap')
print('would mean the random split was letting the model memorize per-client baselines.')

Base rate (is_declining):           30.3%
Random-split AUC (honest features): 0.585
Grouped-split AUC (by client, 5-fold): 0.563
Gap (random - grouped): +0.022

A small gap says client identity isn't doing the model's work for it; a large gap
would mean the random split was letting the model memorize per-client baselines.


In [10]:
# Drop the leaked column and keep only the honest, attacked-and-survived feature set.
assert 'future_impressions_h2' not in model_cols
print('Leaked column removed. Reporting only the honest feature vector going forward:')
print(f'Honest AUC (random split):  {auc_honest:.3f}')
print(f'Honest AUC (grouped split): {auc_grouped:.3f}')

Leaked column removed. Reporting only the honest feature vector going forward:
Honest AUC (random split):  0.585
Honest AUC (grouped split): 0.563


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field(s) | Why |
|---|---|
| `sessions`, `engaged_sessions`, `scroll_events`, and every other `ga4_data_available`-gated column | Same reason as ML-04: coverage is uneven and the flag is three-valued (`TRUE`/`FALSE`/`NULL`) — a blind include would silently treat "not tracked" and "unknown" as "zero engagement" |
| `provider_used`, `model_used` (from `dim_content`, if present) | The data dictionary marks these explicitly "Not a model feature" — they describe how content was *produced*, not how it performs, and using them risks the model learning an authorship artifact instead of a search signal |
| `imp_h2` (H2 impression total) | This is the exact quantity `is_declining` is computed from — the leakage trap in section 3, now removed and never reported |
| `client_hash_id`, `content_hash_id` | Pseudonymous join/grouping keys only — used for the grouped-split test in section 3, never as a model input |
| `fact_content_query_90d` (the whole table) | Not touched this week. Its 90-day window overlaps recent months, and per the data dictionary, using its `impressions_90d`/`*_last30` columns against a label defined on the final month would be leakage — noting this now so it isn't forgotten when the capstone eventually joins it |
| `dim_clients.gsc_data_start` / `ga4_data_start` | Used only for the panel-coverage check in ML-04, not as a feature — a client's history start date describes data availability, not their content's performance |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.